# 48 · Pathway addenda: overlap-respecting external replication and symmetric-class enrichments

Two corrections the internal referee asked for:

| Part | Referee | Question |
|---|---|---|
| A | M7 fix 4 | Do the robust pathways replicate in donor-level external data more than random gene sets with the same sizes, strata **and overlaps**, and more than pathways called only under specimen relabeling? Notebook 37's Mann–Whitney tests treated 36 overlapping pathways as independent. |
| B | D3 residual | The draft still cites notebook 40's pathway enrichments of zonation classes. What are they under notebook 43's symmetric classes, with the same functions? |

The protocol and decision rules were written before any statistic was computed (workstream 5
plan, round 2) and are reproduced in section 1. Cohort: 2 male control mice and 2 cortex sections
from 1 male human donor; external donors are the replicates in part A.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Replication score per set | notebook 37's joint z: matched rank-AUC z of a = sign(our centred interaction) × external Welch t, ÷ √VIF from donor-level residual correlations; exact matched-null moments |
| Collection statistic | mean joint z over the 36 robust (also 26 core) pathways; program version: mean over the 17 programs, each the union of its pathways' informative members |
| Null 1 (primary) | 9,999 stratified gene relabelings within notebook 37's matching strata, applied to every set at once (sizes, strata and all overlaps kept); z and VIF recomputed |
| Null 2 (secondary) | as null 1 with strata also split by our `T_spatial` quintile |
| Comparator | the 21 relabel-called pathways that are not robust (all 28 as sensitivity); D = mean z(robust) − mean z(relabel-called), tested against null 1; program level also by label permutation among programs |
| Secondary statistic | notebook 37's slope-free score (uniform flattening removed on both sides) |
| Decision (Census cortex vs male mice, signed, 36 robust) | **specific**: p_random ≤ 0.01 and p_relabel ≤ 0.05 and program-level label-permutation p ≤ 0.05 · **better than random only**: p_random ≤ 0.01 alone · **not replicated**: p_random > 0.01 |
| Part B | notebook 40's enrichment code unchanged (9,999 draws, seed 40, q ≤ 0.10); class column = notebook 43's `union S2-S1 + S3c-early`; guard: notebook 40's classes reproduce its saved table |
| Keep/change | keep a claim if the same pathway, or one of notebook 40's program for it, is enriched in the same symmetric class |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates' / 'tables'
upstream40 = results_root / 'pt_zonation_classes' / 'tables'
upstream43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_revision_addenda_pathways'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv',
            upstream37 / 'table_S2_pathways.csv', upstream37 / 'gene_level_statistics.csv',
            upstream37 / 'matched_and_joint_tests_all_partitions.csv', upstream37 / 'external_replication_summary.csv',
            upstream40 / 'gene_zonation_classes.csv', upstream40 / 'class_pathway_enrichment.csv',
            upstream43 / 'gene_classes_symmetric.csv',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', lake_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '48.pathway_addenda.1'  # Bump when a cached calculation changes.
ALPHA = .05
SEED, N_PERM = 48, 9999
MIN_CELLS_EXTERNAL = 50       # notebook 37
CONTRIBUTING_QUANTILE, PROGRAM_CUT = .9, .5
SEED40, N_NULL40, ENRICH_Q = 40, 9999, .10   # notebook 40
print('Results folder:', output)

## 2 · Our structures, genes and pathways (notebook 37's construction)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import gene_covariates, matching_strata
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_frame, cached_payload, digest
import pseudospace.set_collection_null as collection_module
import pseudospace.specimen_null as specimen_null_module

pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70)
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(collection_module.__file__), Path(specimen_null_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_all = saved_pt.obs.copy()
saved_pt.file.close()
pt_all = pt_all[pt_all.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}

support = pt_all.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_all.shared_pseudospace.to_numpy(float)
pt_obs = pt_all[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['measured_in_both'] = adata.var.measured_in_both_inputs.to_numpy()
universe['eligible'] = universe.measured_in_both & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
saved_genes = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0).index.tolist()
if genes != saved_genes:
    raise ValueError('The gene universe differs from notebooks 12 and 37.')
Y = expression[:, universe.eligible.to_numpy()]
covariates = universe.set_index('gene').loc[genes]
libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
coverage37 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[universe.measured_in_both, 'gene'], ortholog_map=orthologs, library_name=library,
    tested=genes, min_genes=1) for library in libraries], ignore_index=True)
coverage37['pathway_id'] = coverage37.library + '::' + coverage37.pathway
coverage37 = coverage37[coverage37.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage37.itertuples()}
pathway_names = pd.Series({p: p.split('::', 1)[1] for p in gene_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)

table_s2 = pd.read_csv(upstream37 / 'table_S2_pathways.csv', index_col=0)
robust_ids = table_s2.index[table_s2.robust.astype(bool)].tolist()
core_ids = table_s2.index[table_s2.core.astype(bool)].tolist()
T_spatial = pd.read_csv(upstream37 / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
tests37 = pd.read_csv(upstream37 / 'matched_and_joint_tests_all_partitions.csv')
spatial37 = tests37[tests37.statistic.eq('T_spatial')]
swaps = [p for p in spatial37.partition.unique() if p.startswith('swap')]
relabel_all = sorted(set(spatial37.loc[spatial37.partition.isin(swaps) & spatial37.effect.gt(0) & spatial37.q_normal.le(ALPHA), 'pathway_id']))
relabel_only = [p for p in relabel_all if p not in set(robust_ids)]
print(f'{len(genes):,} genes · {len(gene_sets):,} pathways · robust {len(robust_ids)} · core {len(core_ids)} · '
      f'relabel-called {len(relabel_all)} ({len(relabel_only)} not robust)')

### Programs

The robust programs are notebook 37's 17. The relabel-called pathways are grouped with the same
rule: average linkage on 1 − overlap coefficient of members in the top 10% of `T_spatial`, cut 0.5.

In [ ]:
from pseudospace.pathway_decoys import overlap_programs

contributing = set(T_spatial.index[T_spatial.ge(T_spatial.quantile(CONTRIBUTING_QUANTILE))])
robust_programs = table_s2.loc[robust_ids, 'program'].astype(int)
in_order = lambda ids: [p for p in gene_sets if p in set(ids)]   # notebook 37 clusters in pathway order
check_programs = overlap_programs(gene_sets, in_order(robust_ids), contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program
same_partition = pd.crosstab(robust_programs, check_programs.reindex(robust_ids)).gt(0)
if not (same_partition.sum(axis=0).eq(1).all() and same_partition.sum(axis=1).eq(1).all()):
    raise ValueError('The program rule does not reproduce notebook 37\'s 17 robust programs.')
relabel_programs = overlap_programs(gene_sets, in_order(relabel_only), contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program
print(f'{robust_programs.nunique()} robust programs · {relabel_programs.nunique()} relabel-called programs')

## 3 · External donor-level references (notebook 37's inputs)

Census human cortex (6 donors) and Lake/KPMP healthy cortex (7 donors) against 12 male or 12
female Census mice; per donor, S3 and early (S1 + S2, or convoluted PT) log2 CPM.

In [ ]:
def census_donors(label, segment_map, early_parts):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS_EXTERNAL)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    late, early, info = {}, {}, []

    def log_cpm_of(index):
        return np.log2(table.loc[index].sum() / meta.loc[index, 'total_counts_all_genes'].sum() * 1e6 + 1)

    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        late[donor] = log_cpm_of(block.index[block.segment.eq('S3')])
        early[donor] = log_cpm_of(block.index[block.segment.isin(early_parts)])
        info.append({'donor': donor, 'sex': sex})
    frame = lambda values: pd.DataFrame(values).T.reindex(columns=genes)
    return frame(late), frame(early), pd.DataFrame(info).set_index('donor')


h_late, h_early, _ = census_donors('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                                             'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'])
m_late, m_early, m_info = census_donors('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                  'epithelial cell of proximal tubule segment 2': 'S2',
                                                  'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
males, females = m_info.index[m_info.sex.eq('male')], m_info.index[m_info.sex.eq('female')]


def lake_pseudobulk():
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=genes, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'normal'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': digest(genes),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'sex', 'segment'])
lake_meta['nuclei'] = lake['nuclei']
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS_EXTERNAL).all(axis=1)]
sums_frame = pd.DataFrame(lake['sums'], columns=genes)


def lake_log_cpm(donor, parts):
    rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
    return np.log2(sums_frame.loc[rows].sum() / lake['totals'][rows].sum() * 1e6 + 1)


lake_late = pd.DataFrame({d: lake_log_cpm(d, ['S3']) for d in lake_donors}).T
lake_early = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2']) for d in lake_donors}).T
print(f'Census human {len(h_late)} donors · Lake {len(lake_donors)} donors · mice {len(males)} male, {len(females)} female')

## 4 · Replication scores per reference (notebook 37's `reference_analysis`, up to the pathway test)

In [ ]:
from pseudospace.pathway_decoys import welch_t

short = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
ours_means = {}
for name in names:
    for label, mask in (('S3', short == 'S3'), ('early', np.isin(short, ['S1', 'S2']))):
        ours_means[(name, label)] = np.asarray(Y[(specimen == name) & mask].mean(axis=0)).ravel()
ours_means = pd.DataFrame(ours_means, index=genes)
ours_contrast = pd.DataFrame({name: ours_means[(name, 'S3')] - ours_means[(name, 'early')] for name in names})
ours_raw = ours_contrast[expected['human']].mean(axis=1) - ours_contrast[expected['mouse']].mean(axis=1)
mouse_gradient_ours = ours_contrast[expected['mouse']].mean(axis=1)


def reference_inputs(human_late, human_early, mouse_late, mouse_early, mouse_donors):
    hd = human_late - human_early
    md = (mouse_late - mouse_early).loc[mouse_donors]
    level_h = pd.concat([human_late.mean(), human_early.mean()], axis=1).max(axis=1)
    level_m = pd.concat([mouse_late.loc[mouse_donors].mean(), mouse_early.loc[mouse_donors].mean()], axis=1).max(axis=1)
    keep = (covariates[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & level_h.ge(1) & level_m.ge(1)
            & hd.notna().all() & md.notna().all())
    g = keep.index[keep.to_numpy()]
    centred = ours_contrast - ours_contrast.loc[g].median()
    ours_c = centred[expected['human']].mean(axis=1) - centred[expected['mouse']].mean(axis=1)
    hc, mc = hd.sub(hd[g].median(axis=1), axis=0), md.sub(md[g].median(axis=1), axis=0)
    ext_raw = hd.mean() - md.mean()
    t = pd.Series(welch_t(hc[g].to_numpy(), mc[g].to_numpy())[0], index=g)
    slope_ours = np.polyfit(mouse_gradient_ours.loc[g], ours_raw.loc[g], 1)
    ours_slope_free = ours_raw - np.polyval(slope_ours, mouse_gradient_ours)
    beta_ext = np.polyfit(md[g].mean(), ext_raw.loc[g], 1)[0]
    t_free = pd.Series(welch_t(hc[g].to_numpy(), (1 + beta_ext) * mc[g].to_numpy())[0], index=g)
    strata_g = matching_strata(pd.DataFrame({'mean_expression': ((level_h + level_m) / 2).loc[g].to_numpy(),
                                             'detection': covariates.loc[g, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                             'coverage': covariates.loc[g, 'coverage'].to_numpy()}), bins=3)
    sets_g = {p: [x for x in v if x in set(g)] for p, v in gene_sets.items()}
    sets_g = {p: v for p, v in sets_g.items() if 5 <= len(v) < len(g)}
    residuals = pd.concat([hc[g] - hc[g].mean(), mc[g] - mc[g].mean()])
    return {'genes': g, 'signed': np.sign(ours_c.loc[g]) * t, 'slope-free': np.sign(ours_slope_free.loc[g]) * t_free,
            'strata': strata_g, 'sets': sets_g, 'residuals': residuals.to_numpy()}


slugs = {'Census cortex − male mouse (primary)': 'census_male', 'Lake/KPMP cortex − male mouse': 'lake_male',
         'Census cortex − female mouse': 'census_female', 'Lake/KPMP cortex − female mouse': 'lake_female'}
references = {'Census cortex − male mouse (primary)': (h_late, h_early, m_late, m_early, males),
              'Lake/KPMP cortex − male mouse': (lake_late, lake_early, m_late, m_early, males),
              'Census cortex − female mouse': (h_late, h_early, m_late, m_early, females),
              'Lake/KPMP cortex − female mouse': (lake_late, lake_early, m_late, m_early, females)}
reference_data = {label: reference_inputs(*spec) for label, spec in references.items()}
display(pd.DataFrame({label: {'informative genes': len(r['genes']), 'testable pathways': len(r['sets']),
                              'robust testable': sum(p in r['sets'] for p in robust_ids),
                              'relabel-called testable': sum(p in r['sets'] for p in relabel_only)}
                      for label, r in reference_data.items()}))

### Sets scored per reference, and a guard against notebook 37

Pathway sets: the robust and relabel-called pathways (all 28). Program sets: the union of each
program's informative members. The guard compares the exact-moment joint z of the robust pathways
with notebook 37's saved primary replication z (9,999 random sets).

In [ ]:
from pseudospace.set_collection_null import (JointReplicationZ, collection_null, label_permutation_p,
                                             upper_p)


def scored_sets(ref):
    sets_g = ref['sets']
    pathways = {p: sets_g[p] for p in sorted(set(robust_ids) | set(relabel_all)) if p in sets_g}
    programs = {}
    for kind, assignment in (('robust', robust_programs), ('relabel', relabel_programs)):
        for program, ids in assignment.groupby(assignment).groups.items():
            union = sorted({g for p in ids if p in sets_g for g in sets_g[p]})
            if len(union) >= 5:
                programs[f'program:{kind}:{int(program)}'] = union
    return {**pathways, **programs}


def collections_of(sets_):
    keep = lambda ids: [p for p in ids if p in sets_]
    return {'robust (36)': keep(robust_ids), 'core (26)': keep(core_ids), 'relabel-called, not robust (21)': keep(relabel_only),
            'relabel-called, all (28)': keep(relabel_all),
            'robust programs (17)': [p for p in sets_ if p.startswith('program:robust:')],
            'relabel-called programs': [p for p in sets_ if p.startswith('program:relabel:')]}


primary_label = 'Census cortex − male mouse (primary)'
primary_ref = reference_data[primary_label]
primary_sets = scored_sets(primary_ref)
primary_scorer = JointReplicationZ(primary_ref['signed'], primary_ref['strata'], primary_sets, primary_ref['residuals'])
guard = pd.DataFrame({'exact moments': pd.Series(primary_scorer.z(), index=primary_scorer.ids).reindex(robust_ids),
                      'notebook 37': table_s2.loc[robust_ids, 'external_z']})
guard_diff = (guard['exact moments'] - guard['notebook 37']).abs()
print(f'Guard: robust-pathway replication z, exact moments vs notebook 37: median |Δ| {guard_diff.median():.3f}, '
      f'max {guard_diff.max():.3f}, r = {guard.corr().iloc[0, 1]:.4f}')
if guard_diff.max() > .3 or guard.corr().iloc[0, 1] < .99:
    raise ValueError('The replication score does not reproduce notebook 37.')

## 5 · Overlap-preserving permutation tests

For every reference, statistic (signed, slope-free) and null (matched strata, or matched strata ×
`T_spatial` quintile), 9,999 gene relabelings are drawn within strata and applied to every scored
set at once. Each draw rescores all sets (matched z and VIF) and records each collection's mean.

In [ ]:
def run_null(label, statistic, null):
    ref = reference_data[label]
    sets_ = scored_sets(ref)
    strata = ref['strata']
    if null == 'selection-matched':
        quintile = pd.qcut(T_spatial.loc[ref['genes']].rank(method='first'), 5, labels=False).to_numpy()
        strata = strata * 5 + quintile
    scorer = JointReplicationZ(ref[statistic], strata, sets_, ref['residuals'])
    observed, draws = collection_null(scorer, collections_of(sets_), strata, N_PERM, SEED)
    return {'observed': observed.to_numpy(), 'draws': draws.to_numpy(), 'collections': np.array(list(observed.index), dtype=str),
            'set_ids': np.array(scorer.ids, dtype=str), 'set_z': scorer.z()}


runs = {}
for label in reference_data:
    for statistic in ('signed', 'slope-free'):
        for null in ('matched', 'selection-matched'):
            ref = reference_data[label]
            runs[(label, statistic, null)] = cached_payload(
                f'collection_null_{slugs[label]}_{statistic}_{null}',
                lambda label=label, statistic=statistic, null=null: run_null(label, statistic, null), root=cache,
                params={'n_perm': N_PERM, 'seed': SEED, 'label': label, 'statistic': statistic, 'null': null, 'payload': 'str-arrays'},
                inputs={'statistic': digest(ref[statistic]), 'strata': ref['strata'], 'T_spatial': digest(T_spatial),
                        'sets': digest(sorted(scored_sets(ref).items()))}, code=code)

rows = []
for (label, statistic, null), run in runs.items():
    collections = list(run['collections'])
    observed = pd.Series(run['observed'], index=collections)
    draws = pd.DataFrame(run['draws'], columns=collections)
    set_z = pd.Series(run['set_z'], index=run['set_ids'])
    row = {'reference': label, 'statistic': statistic, 'null': null}
    for name in collections:
        row[f'mean z, {name}'] = observed[name]
        row[f'null median, {name}'] = draws[name].median()
    for name in ('robust (36)', 'core (26)', 'robust programs (17)', 'relabel-called, not robust (21)'):
        row[f'p_random, {name}'] = upper_p(observed[name], draws[name])
    for comparator, tag in (('relabel-called, not robust (21)', '21'), ('relabel-called, all (28)', '28')):
        diff = observed['robust (36)'] - observed[comparator]
        row[f'D vs relabel {tag}'] = diff
        row[f'p_relabel {tag}'] = upper_p(diff, draws['robust (36)'] - draws[comparator])
    diff_prog = observed['robust programs (17)'] - observed['relabel-called programs']
    row['D programs'] = diff_prog
    row['p_relabel programs (gene relabeling)'] = upper_p(diff_prog, draws['robust programs (17)'] - draws['relabel-called programs'])
    robust_prog_z = set_z[[i for i in set_z.index if i.startswith('program:robust:')]]
    relabel_prog_z = set_z[[i for i in set_z.index if i.startswith('program:relabel:')]]
    row['p_relabel programs (label permutation)'] = label_permutation_p(robust_prog_z, relabel_prog_z, seed=SEED)[0]
    row['robust programs / relabel programs'] = f'{len(robust_prog_z)} / {len(relabel_prog_z)}'
    rows.append(row)
replication = pd.DataFrame(rows)
replication.to_csv(tables / 'table_replication_overlap_null.csv', index=False)
key_columns = ['reference', 'statistic', 'null', 'mean z, robust (36)', 'null median, robust (36)', 'p_random, robust (36)',
               'p_random, core (26)', 'p_random, robust programs (17)', 'mean z, relabel-called, not robust (21)',
               'p_random, relabel-called, not robust (21)', 'D vs relabel 21', 'p_relabel 21', 'p_relabel 28',
               'D programs', 'p_relabel programs (gene relabeling)', 'p_relabel programs (label permutation)']
display(replication[key_columns].round(4))
per_set = pd.concat([pd.DataFrame({'set': run['set_ids'], 'z_joint': run['set_z']}).assign(reference=label, statistic=statistic)
                     for (label, statistic, null), run in runs.items() if null == 'matched'], ignore_index=True)
per_set['name'] = per_set.set.map(lambda s: pathway_names.get(s, s))
per_set['group'] = np.select([per_set.set.isin(robust_ids), per_set.set.isin(relabel_only), per_set.set.isin(relabel_all),
                              per_set.set.str.startswith('program:robust'), per_set.set.str.startswith('program:relabel')],
                             ['robust', 'relabel-called', 'relabel-called and robust', 'robust program', 'relabel-called program'], 'other')
per_set.to_csv(tables / 'replication_z_per_set.csv', index=False)

### Notebook 37's Mann–Whitney p-values, for comparison

These are the values the referee asked to replace: they treat the 36 overlapping pathways as
independent observations.

In [ ]:
summary37 = pd.read_csv(upstream37 / 'external_replication_summary.csv', index_col=0)
display(summary37.loc[[i for i in summary37.index if 'Mann' in i or 'median replication' in i]])

## 6 · Pre-specified decision for R4

In [ ]:
primary_row = replication[(replication.reference.eq(primary_label)) & replication.statistic.eq('signed')
                          & replication.null.eq('matched')].iloc[0]
beats_random = primary_row['p_random, robust (36)'] <= .01
beats_relabel = primary_row['p_relabel 21'] <= .05 and primary_row['p_relabel programs (label permutation)'] <= .05
REPLICATION_DECISION = ('specific' if beats_random and beats_relabel else
                        'better than random only' if beats_random else 'not replicated')
lake_row = replication[(replication.reference.eq('Lake/KPMP cortex − male mouse')) & replication.statistic.eq('signed')
                       & replication.null.eq('matched')].iloc[0]
lake_decision = ('specific' if lake_row['p_random, robust (36)'] <= .01 and lake_row['p_relabel 21'] <= .05
                 and lake_row['p_relabel programs (label permutation)'] <= .05 else
                 'better than random only' if lake_row['p_random, robust (36)'] <= .01 else 'not replicated')
decision = pd.Series({'p_random, 36 robust (primary)': primary_row['p_random, robust (36)'],
                      'p_relabel, 21 pathways (gene relabeling)': primary_row['p_relabel 21'],
                      'p_relabel, programs (label permutation)': primary_row['p_relabel programs (label permutation)'],
                      'decision (Census cortex, male mice)': REPLICATION_DECISION,
                      'same rule, Lake cortex, male mice': lake_decision})
decision.to_csv(tables / 'decision_replication.csv')
display(decision.to_frame('value'))

## 7 · Part B · Notebook 40's class-enrichment pipeline

Notebook 40's inputs are rebuilt unchanged: all PT structures (no coordinate filter), accepted
orthologs measured in all four specimens, cortical-like mouse S3 from glomerulus depth (centroids
verified against the fine GeoJSON), the 9,029 classified genes, pathways with 10–300 classified
members, matching strata (mean expression, detection, minimum species detection), and the CAMERA
inflation from residual correlations after removing specimen × segment means. The enrichment is run
twice: with notebook 40's classes (a guard that must reproduce its saved table) and with notebook
43's symmetric classes.

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation
from pseudospace.pathway_remodeling import matched_pathway_tests, residual_pathway_correlations, correlation_adjusted_rank_tests
from pseudospace.pathway_decoys import joint_matched_test

adata40 = rebuild_pt_expression(pt_all, source_paths, orthologs, target_sum=1e4)
measured40 = adata40.var.measured_in_both_inputs.to_numpy()
gene_space = adata40.var_names[measured40]
lognorm40 = sparse.csr_matrix(adata40.layers['lognorm'], dtype=float)[:, measured40]
specimen40 = adata40.obs['sample'].astype(str).to_numpy()
segment40 = adata40.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen40), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen40 == name
        structures = source_obs.loc[adata40.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata40.obs_names)}, code=code)['depth']
cortical_s3 = np.zeros(len(specimen40), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen40 == name) & (segment40 == 'S1')], .95)
    cortical_s3 |= (specimen40 == name) & (segment40 == 'S3') & (depth <= limit)
segment_region = np.where((segment40 == 'S3') & np.isin(specimen40, expected['mouse']) & ~cortical_s3, 'S3deep', segment40)

classes40 = pd.read_csv(upstream40 / 'gene_zonation_classes.csv', index_col=0)['class']
classes43 = pd.read_csv(upstream43 / 'gene_classes_symmetric.csv', index_col=0)['union S2-S1 + S3c-early']
universe_genes = classes40.index[classes40.notna()]
if not classes43.index[classes43.notna()].equals(universe_genes):
    raise ValueError('Notebooks 40 and 43 classify different genes.')
membership40 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    gene_space, ortholog_map=orthologs, library_name=library, tested=list(universe_genes), min_genes=1)
    for library in libraries], ignore_index=True)
membership40['pathway_id'] = membership40.library + '::' + membership40.pathway
membership40 = membership40[membership40.n_tested.between(10, 300)]
class_sets = {row.pathway_id: [g for g in row.genes_present if g in set(universe_genes)] for row in membership40.itertuples()}
class_sets = {p: v for p, v in class_sets.items() if 10 <= len(v) <= 300}
class_names = pd.Series({p: p.split('::', 1)[1] for p in class_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)
structure_values = lognorm40[:, gene_space.get_indexer(universe_genes)]
mean_expression = pd.Series(np.asarray(structure_values.mean(axis=0)).ravel(), index=universe_genes)
detection = pd.Series(np.asarray((structure_values > 0).mean(axis=0)).ravel(), index=universe_genes)
min_detection = pd.Series(np.minimum(*[np.asarray((structure_values[np.isin(specimen40, expected[s])] > 0).mean(axis=0)).ravel()
                                        for s in ('mouse', 'human')]), index=universe_genes)
covariates40 = pd.DataFrame({'mean_expression': mean_expression, 'detection': detection, 'coverage': min_detection})
strata3, strata5 = matching_strata(covariates40, bins=3), matching_strata(covariates40, bins=5)


def class_residual_correlations():
    dense = structure_values.toarray()
    groups = pd.Series(pd.Series(specimen40).astype(str) + '|' + pd.Series(segment_region).astype(str))
    for _, idx in groups.groupby(groups).groups.items():
        rows_ = np.asarray(idx)
        dense[rows_] -= dense[rows_].mean(axis=0)
    return residual_pathway_correlations(dense, specimen40, list(universe_genes), class_sets)


class_correlations = cached_frame('class_residual_correlations', class_residual_correlations, root=cache,
                                  inputs={'genes': list(universe_genes), 'sets': digest(sorted(class_sets.items())),
                                          'structures': len(specimen40)}, code=code)
upper = mean_expression.index[mean_expression.ge(mean_expression.quantile(1 / 3))]
upper_sets = {p: [g for g in v if g in set(upper)] for p, v in class_sets.items()}
upper_sets = {p: v for p, v in upper_sets.items() if len(v) >= 10}
CLASS_LABELS = ('conserved', 'reversal', 'mouse-only', 'human-only')


ALL_INPUTS = {'universe': universe_genes, 'sets': class_sets, 'strata3': strata3, 'strata5': strata5, 'upper': upper,
              'upper_sets': upper_sets, 'correlations': class_correlations}


def class_enrichment(classes, tag, inputs=ALL_INPUTS):
    """Notebook 40, section 7, for one class assignment and one classified-gene universe."""
    universe_, sets_all = inputs['universe'], inputs['sets']
    indicator = pd.DataFrame({k: classes.loc[universe_].eq(k).astype(float) for k in CLASS_LABELS})
    vif = correlation_adjusted_rank_tests(indicator['conserved'], sets_all, inputs['correlations']).set_index('pathway_id').variance_inflation
    enrich = {}
    for label, frame_, sets_, strata_ in (('primary', indicator, sets_all, inputs['strata3']),
                                          ('5-bin', indicator, sets_all, inputs['strata5']),
                                          ('upper expression', indicator.loc[inputs['upper']], inputs['upper_sets'],
                                           pd.Series(inputs['strata3'], index=universe_).loc[inputs['upper']].to_numpy())):
        result = cached_frame(f'class_enrichment_{tag}_{label.replace(" ", "_")}',
                              lambda frame_=frame_, sets_=sets_, strata_=strata_:
                              matched_pathway_tests(frame_, sets_, strata_, n_null=N_NULL40, seed=SEED40), root=cache,
                              inputs={'indicator': frame_, 'sets': digest(sorted(sets_.items())), 'strata': strata_}, code=code)
        enrich[label] = joint_matched_test(result, vif, keys=('statistic',))
    table = enrich['primary'].copy()
    for label, column in (('5-bin', 'q_5bin'), ('upper expression', 'q_upper_expression')):
        table = table.merge(enrich[label][['pathway_id', 'statistic', 'q_joint']].rename(columns={'q_joint': column}),
                            on=['pathway_id', 'statistic'], how='left')
    table['name'] = table.pathway_id.map(class_names)
    table['class_members'] = [int(indicator.loc[sets_all[p], c].sum()) for p, c in zip(table.pathway_id, table.statistic)]
    table['class_genes'] = [', '.join(g for g in sets_all[p] if indicator.loc[g, c] == 1) for p, c in zip(table.pathway_id, table.statistic)]
    table['enriched'] = table.effect.gt(0) & table.q_joint.le(ENRICH_Q)
    table['not_expression_driven'] = table.enriched & table.q_5bin.le(ENRICH_Q) & table.q_upper_expression.le(ENRICH_Q)
    groups_ = []
    for klass in CLASS_LABELS:
        ids = table.loc[table.statistic.eq(klass) & table.enriched, 'pathway_id'].tolist()
        if ids:
            members = set(indicator.index[indicator[klass].eq(1)])
            groups_.append(overlap_programs(sets_all, ids, contributing=members, cut=.5).assign(statistic=klass))
    groups_ = pd.concat(groups_, ignore_index=True) if groups_ else pd.DataFrame(columns=['pathway_id', 'program', 'statistic'])
    return table.merge(groups_, on=['pathway_id', 'statistic'], how='left'), indicator


enrichment40, _ = class_enrichment(classes40, 'nb40classes')
saved40 = pd.read_csv(upstream40 / 'class_pathway_enrichment.csv')
merged = enrichment40.merge(saved40[['pathway_id', 'statistic', 'q_joint', 'enriched']], on=['pathway_id', 'statistic'], suffixes=('', '_saved'))
if len(merged) != len(saved40) or not np.allclose(merged.q_joint, merged.q_joint_saved, rtol=1e-6) \
        or not merged.enriched.eq(merged.enriched_saved.astype(bool)).all():
    raise ValueError('The guard failed: notebook 40\'s classes do not reproduce its saved enrichment table.')
print(f'Guard passed: {len(merged):,} rows reproduce notebook 40 (enriched {int(merged.enriched.sum())}).')

enrichment43, indicator43 = class_enrichment(classes43, 'symmetric')
enrichment43.sort_values(['statistic', 'q_joint']).to_csv(tables / 'class_pathway_enrichment_symmetric.csv', index=False)
summary_classes = pd.DataFrame({
    'genes, notebook 40': classes40.loc[universe_genes].value_counts().reindex(CLASS_LABELS),
    'genes, symmetric': classes43.loc[universe_genes].value_counts().reindex(CLASS_LABELS),
    'enriched pathways, notebook 40': saved40.groupby('statistic').enriched.sum().reindex(CLASS_LABELS),
    'enriched pathways, symmetric': enrichment43.groupby('statistic').enriched.sum().reindex(CLASS_LABELS),
    'not expression-driven, notebook 40': saved40.groupby('statistic').not_expression_driven.sum().reindex(CLASS_LABELS),
    'not expression-driven, symmetric': enrichment43.groupby('statistic').not_expression_driven.sum().reindex(CLASS_LABELS)})
summary_classes.to_csv(tables / 'class_enrichment_counts.csv')
display(summary_classes)
display(enrichment43[enrichment43.enriched].sort_values(['statistic', 'q_joint'])[
    ['statistic', 'name', 'class_members', 'n_genes', 'q_joint', 'q_5bin', 'q_upper_expression', 'not_expression_driven', 'program']])

### Probe-balanced symmetric classes, with the declared naming rule (round 2b)

The draft's class counts (41 mouse-only, 76 human-only) use probe-balanced genes. This rerun
restricts the classified universe to them and recomputes strata, residual correlations and VIF. The
rule was declared before this run but after the all-gene run above had been seen (plan, round 2b).
- An enrichment counts only if q ≤ 0.10, it is not expression-driven, and ≥ 3 class genes carry it.
- If neither species-only class has one, no pathway is named for species-only zonation and
  Figure S8d is dropped.

In [ ]:
balanced_flag = pd.read_csv(upstream43 / 'gene_classes_symmetric.csv', index_col=0)['probe_balanced'].astype(bool)
balanced_genes = universe_genes[balanced_flag.reindex(universe_genes).fillna(False).to_numpy()]
sets_b = {p: [g for g in v if g in set(balanced_genes)] for p, v in class_sets.items()}
sets_b = {p: v for p, v in sets_b.items() if 10 <= len(v) <= 300}
expression_b = mean_expression.loc[balanced_genes]
upper_b = expression_b.index[expression_b.ge(expression_b.quantile(1 / 3))]
upper_sets_b = {p: [g for g in v if g in set(upper_b)] for p, v in sets_b.items()}
upper_sets_b = {p: v for p, v in upper_sets_b.items() if len(v) >= 10}


def balanced_residual_correlations():
    dense = structure_values[:, universe_genes.get_indexer(balanced_genes)].toarray()
    groups = pd.Series(pd.Series(specimen40).astype(str) + '|' + pd.Series(segment_region).astype(str))
    for _, idx in groups.groupby(groups).groups.items():
        rows_ = np.asarray(idx)
        dense[rows_] -= dense[rows_].mean(axis=0)
    return residual_pathway_correlations(dense, specimen40, list(balanced_genes), sets_b)


inputs_b = {'universe': balanced_genes, 'sets': sets_b, 'strata3': matching_strata(covariates40.loc[balanced_genes], bins=3),
            'strata5': matching_strata(covariates40.loc[balanced_genes], bins=5), 'upper': upper_b, 'upper_sets': upper_sets_b,
            'correlations': cached_frame('class_residual_correlations_balanced', balanced_residual_correlations, root=cache,
                                         inputs={'genes': list(balanced_genes), 'sets': digest(sorted(sets_b.items())),
                                                 'structures': len(specimen40)}, code=code)}
enrichment_b, _ = class_enrichment(classes43, 'symmetric_balanced', inputs_b)
enrichment_b.sort_values(['statistic', 'q_joint']).to_csv(tables / 'class_pathway_enrichment_symmetric_probe_balanced.csv', index=False)
print('Probe-balanced class sizes:', classes43.loc[balanced_genes].value_counts().reindex(CLASS_LABELS).to_dict())
naming_rows = []
for run, table_ in (('all classified genes', enrichment43), ('probe-balanced', enrichment_b)):
    for klass in ('mouse-only', 'human-only', 'conserved', 'reversal'):
        block_ = table_[table_.statistic.eq(klass) & table_.enriched]
        counting = block_[block_.not_expression_driven & block_.class_members.ge(3)]
        naming_rows.append({'run': run, 'class': klass, 'enriched (q ≤ 0.10)': len(block_),
                            'not expression-driven': int(block_.not_expression_driven.sum()),
                            'counting (also ≥ 3 class genes)': len(counting),
                            'counting pathways': '; '.join(f"{r['name']} ({r['class_genes']})" for _, r in counting.iterrows())})
naming = pd.DataFrame(naming_rows)
naming.to_csv(tables / 'declared_naming_rule.csv', index=False)
display(naming)
species_only_b = naming[naming.run.eq('probe-balanced') & naming['class'].isin(['mouse-only', 'human-only'])]
NAMING_DECISION = ('no pathway named for species-only zonation; drop Figure S8d'
                   if species_only_b['counting (also ≥ 3 class genes)'].sum() == 0 else
                   'name only: ' + ' | '.join(f"{r['class']}: {r['counting pathways']}" for _, r in species_only_b.iterrows()
                                              if r['counting (also ≥ 3 class genes)'] > 0))
print('Declared rule (probe-balanced):', NAMING_DECISION)

### Integration genes by class (notebook 42's check, symmetric classes)

In [ ]:
pass2 = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
         if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if not pass2:
    raise FileNotFoundError('Notebook 13 pass-2 embedding not found in its stage cache.')
pass2_var = ad.read_h5ad(str(pass2[0])[:-len('.key.json')], backed='r').var
hvg752 = set(pass2_var.index[pass2_var.highly_variable_for_harmony.to_numpy(bool)])
pca611 = set(pass2_var.index[(pass2_var.highly_variable_for_harmony & pass2_var.highly_variable).to_numpy(bool)])
integration_rows = []
for version, classes in (('notebook 40', classes40), ('symmetric', classes43)):
    for set_label, gene_set in (('611 PCA genes', pca611), ('752 Harmony HVGs', hvg752)):
        inside = classes.loc[classes.index.intersection(sorted(gene_set))].value_counts()
        totals = classes.value_counts()
        for klass in [*CLASS_LABELS, 'indeterminate', 'neither']:
            integration_rows.append({'classes': version, 'gene set': set_label, 'class': klass,
                                     'genes in set': int(inside.get(klass, 0)), 'class size': int(totals.get(klass, 0)),
                                     'share of class': inside.get(klass, 0) / totals.get(klass, np.nan)})
integration = pd.DataFrame(integration_rows)
integration.to_csv(tables / 'integration_genes_by_class.csv', index=False)
display(integration.pivot_table(index=['gene set', 'class'], columns='classes', values='share of class').round(3))

## 8 · Replacement table for the draft's class claims

A claim is kept if the same pathway, or another pathway of notebook 40's program for it, is
enriched (q ≤ 0.10) in the same symmetric class. Class sizes changed (for example mouse-only 285 →
51 genes), so power differs: a missing enrichment is "not detected", not absence.

In [ ]:
claims = [('Draft lines 613–614: mouse-only genes enriched for sterol synthesis', 'mouse-only', 'KEGG_2019_Mouse::Steroid biosynthesis'),
          ('Draft lines 613–614: mouse-only genes enriched for peroxisomal β-oxidation', 'mouse-only',
           'Reactome_2022::Beta-oxidation Of Very Long Chain Fatty Acids R-HSA-390247'),
          ('Draft lines 613–614: mouse-only genes enriched for tyrosine catabolism', 'mouse-only', 'KEGG_2019_Mouse::Tyrosine metabolism'),
          ('Notebook 40 text: mouse-only genes enriched for pyrimidine catabolism', 'mouse-only', 'KEGG_2019_Mouse::Pyrimidine metabolism')]
for _, row in pd.read_csv(upstream40 / 'class_programs.csv').iterrows():
    match = saved40[saved40.statistic.eq(row['class']) & saved40.program.eq(row['program']) & saved40.name.eq(row['name'])]
    if row['class'] != 'mouse-only' and len(match):
        claims.append((f"Figure S8d: {row['class']} programs include {row['name']}", row['class'], match.pathway_id.iloc[0]))
replacement = []
for claim, klass, pathway in claims:
    old = saved40[saved40.statistic.eq(klass) & saved40.pathway_id.eq(pathway)]
    if old.empty:
        raise ValueError(f'{pathway} is not in notebook 40\'s enrichment table.')
    old = old.iloc[0]
    program_ids = saved40.loc[saved40.statistic.eq(klass) & saved40.program.eq(old.program), 'pathway_id'].tolist()
    new = enrichment43[enrichment43.statistic.eq(klass)].set_index('pathway_id')
    same_program_hits = [p for p in program_ids if p in new.index and new.loc[p, 'enriched']]
    balanced_row = enrichment_b[enrichment_b.statistic.eq(klass) & enrichment_b.pathway_id.eq(pathway)]
    replacement.append({'claim': claim, 'class': klass, 'pathway': class_names[pathway],
                        'probe-balanced: q (class members)': (f"{balanced_row.q_joint.iloc[0]:.2g} ({int(balanced_row.class_members.iloc[0])})"
                                                              if len(balanced_row) else 'not testable'),
                        'old: q (class members)': f"{old.q_joint:.2g} ({int(old.class_members)})",
                        'new: q (class members)': f"{new.loc[pathway, 'q_joint']:.2g} ({int(new.loc[pathway, 'class_members'])})",
                        'new: enriched in program': '; '.join(class_names[p] for p in same_program_hits),
                        'new: not expression-driven': bool(new.loc[pathway, 'not_expression_driven']),
                        'decision': 'keep' if same_program_hits else 'change'})
share = integration.set_index(['classes', 'gene set', 'class'])['share of class']
replacement.insert(0, {'claim': 'Draft lines 311–312: integration genes are 16% of conserved and 2% of mouse-only genes',
                       'class': 'conserved / mouse-only', 'pathway': '611 PCA genes',
                       'old: q (class members)': f"{share[('notebook 40', '611 PCA genes', 'conserved')]:.0%} / "
                                                 f"{share[('notebook 40', '611 PCA genes', 'mouse-only')]:.0%}",
                       'new: q (class members)': f"{share[('symmetric', '611 PCA genes', 'conserved')]:.0%} / "
                                                 f"{share[('symmetric', '611 PCA genes', 'mouse-only')]:.0%}",
                       'new: enriched in program': '', 'new: not expression-driven': np.nan,
                       'decision': 'change numbers'})
replacement = pd.DataFrame(replacement)
replacement.to_csv(tables / 'table_class_claim_replacements.csv', index=False)
display(replacement)

## 8b · AKI strategies under the paper's R2 rule and notebook 45's D6 rule (round 2b)

- **R2 rule** (Methods 6.14, notebook 31): NCDR < 0.25 **and** mean relabeled calls < 5% of
  tested pathways.
- **D6 rule** (notebook 45): mean relabeled calls ≤ 5% of condition calls.

The numbers come from notebook 45's saved tables; nothing is recomputed. Our cohort's `T_spatial`
NCDR (notebook 37) and the within-group specimen × segment coherence (notebook 45) are shown
beside them.

In [ ]:
method45 = results_root / 'pt_revision_method'
aki45 = pd.read_csv(method45 / 'tables' / 'table_aki_strategy_calls.csv', index_col=0)
record45 = json.loads((method45 / 'run_record.json').read_text())
tested_aki = int(record45['pools']['N6 our AKI design']['pathways'])
relabel_cols45 = [c for c in aki45.columns if c.startswith('relabel_')]
aki_rules = pd.DataFrame({'condition calls': aki45.condition, 'relabeled calls': aki45[relabel_cols45].astype(int).astype(str).agg(', '.join, axis=1),
                          'mean relabeled': aki45[relabel_cols45].mean(axis=1)})
aki_rules['NCDR'] = aki_rules['mean relabeled'] / aki_rules['condition calls']
aki_rules['relabeled ÷ tested'] = aki_rules['mean relabeled'] / tested_aki
aki_rules['passes R2 rule'] = aki_rules.NCDR.lt(.25) & aki_rules['relabeled ÷ tested'].lt(.05)
aki_rules['passes D6 rule'] = aki_rules.NCDR.le(.05)
aki_rules.to_csv(tables / 'aki_strategies_under_both_rules.csv')
display(aki_rules.round(3))
table_s1 = pd.read_csv(upstream37 / 'table_S1_strategy_specificity.csv', index_col=0)
coherence45 = pd.read_csv(method45 / 'tables' / 'table_mechanism_coherence.csv', index_col=0)
margin = pd.Series({'tested pathways (AKI)': tested_aki,
                    'our cohort T_spatial joint NCDR (notebook 37)': table_s1.loc['8 · Gene GAM × smooth position (T_spatial) (joint test)', 'ncdr'],
                    'our cohort within-species specimen × segment coherence': coherence45.loc['N7 our cohort', 'median ρ, specimen × segment'],
                    'AKI within-condition specimen × segment coherence': coherence45.loc['N6 our AKI design', 'median ρ, specimen × segment'],
                    'human pools specimen × segment coherence (min)': coherence45.loc[['N3 Lake/KPMP human cortex', 'N4 Census human cortex'],
                                                                                       'median ρ, specimen × segment'].min()})
margin.to_csv(tables / 'aki_rule_margin_context.csv')
display(margin.to_frame('value'))

## 8c · Our cohort's step screen: 50 (notebook 45) against 16 (notebook 37) species calls

Both tests are joint tests of group × segment steps beyond a group offset. They differ in the
shared shape:
- **notebook 37:** a 6-df spline in the scFates coordinate, with the species' smooth shape as
  the relabeling nuisance;
- **notebook 45:** segment intercepts only, with species × segment as the nuisance, so no
  coordinate is used.

They also differ in the VIF source (spline-fit residuals against specimen × segment-centred
residuals) and in the matched null (9,999 sets against exact moments). The call sets are compared
here; notebook 45's calls are read from its stage cache and checked against its saved count.

In [ ]:
import glob as _glob

anchor_counts = pd.read_csv(method45 / 'tables' / 'anchor_our_cohort.csv', index_col=0)
expected_p1 = int(anchor_counts.loc['P1 T_steps joint', 'species'])
candidates = []
for path in sorted(_glob.glob(str(method45 / 'stage_cache' / 'anchor_our_cohort__*.csv'))):
    frame = pd.read_csv(path)
    row = frame[frame.kind.eq('summary') & frame.strategy.eq('P1 T_steps joint') & frame.label.eq('species')]
    if len(row) and int(row.calls.iloc[0]) == expected_p1:
        candidates.append(tuple(int(i) for i in str(row.called.iloc[0]).split('|') if i not in ('', 'nan')))
if not candidates or len(set(candidates)) != 1:
    raise ValueError('Notebook 45\'s step-screen calls for our cohort could not be read unambiguously.')
# Notebook 45 indexed pathways in its own order: membership built on the tested genes, which
# `build_pathway_membership` sorts by assayed size, so the order differs from notebook 37's.
order45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    genes, ortholog_map=orthologs, library_name=library, tested=genes, min_genes=1) for library in libraries], ignore_index=True)
order45 = (order45.library + '::' + order45.pathway)[order45.n_tested.between(10, min(300, len(genes) - 1))].tolist()
if sorted(order45) != sorted(gene_sets):
    raise ValueError('Notebook 45 and notebook 37 pathway families differ.')
steps45 = {order45[i] for i in candidates[0]}
species37 = tests37[tests37.partition.eq('species')]
steps37 = set(species37.loc[species37.statistic.eq('T_discrete_spatial') & species37.effect.gt(0) & species37.q_joint.le(ALPHA), 'pathway_id'])
beyond37 = set(species37.loc[species37.statistic.eq('T_position_given_segments') & species37.effect.gt(0) & species37.q_joint.le(ALPHA), 'pathway_id'])
spatial_calls37 = set(species37.loc[species37.statistic.eq('T_spatial') & species37.effect.gt(0) & species37.q_joint.le(ALPHA), 'pathway_id'])
reconcile = pd.Series({'notebook 45 steps (coordinate-free)': len(steps45), 'notebook 37 steps (spline-based)': len(steps37),
                       'notebook 37 steps also called by notebook 45': len(steps37 & steps45),
                       'robust (36) among notebook 45 steps': len(set(robust_ids) & steps45),
                       'robust (36) among notebook 37 steps': len(set(robust_ids) & steps37),
                       'robust (36) among notebook 37 smooth-beyond-steps calls': len(set(robust_ids) & beyond37),
                       'notebook 37 T_spatial candidates (60) among notebook 45 steps': len(spatial_calls37 & steps45),
                       'median notebook 37 step q of notebook 45 calls': float(species37[species37.statistic.eq('T_discrete_spatial')]
                                                                                 .set_index('pathway_id').loc[sorted(steps45), 'q_joint'].median())})
reconcile.to_csv(tables / 'step_screen_reconciliation.csv')
display(reconcile.to_frame('value'))

## 9 · Figure: replication against overlap-preserving nulls

In [ ]:
import matplotlib.pyplot as plt

ROBUST, RELABEL, MUTED, INK = '#2a78d6', '#eb6834', '#8a8984', '#0b0b0b'
plt.rcParams.update({'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': MUTED})
panels = [(primary_label, 'signed', 'matched'), ('Lake/KPMP cortex − male mouse', 'signed', 'matched'),
          (primary_label, 'signed', 'selection-matched'), (primary_label, 'slope-free', 'matched')]
fig, axes = plt.subplots(1, len(panels), figsize=(10, 2.8), sharey=False)
for ax, key in zip(axes, panels):
    run = runs[key]
    collections = list(run['collections'])
    draws = pd.DataFrame(run['draws'], columns=collections)
    observed = pd.Series(run['observed'], index=collections)
    for name, color in (('robust (36)', ROBUST), ('relabel-called, not robust (21)', RELABEL)):
        ax.hist(draws[name], bins=50, color=color, alpha=.35, density=True, label=f'null, {name.split(",")[0]}')
        ax.axvline(observed[name], color=color, lw=2)
    row = replication[(replication.reference.eq(key[0])) & replication.statistic.eq(key[1]) & replication.null.eq(key[2])].iloc[0]
    ax.set_title(f"{key[0].split(' −')[0]}, {key[1]}, {key[2]}\np_random {row['p_random, robust (36)']:.4f} · "
                 f"p_relabel {row['p_relabel 21']:.3f}", fontsize=7, loc='left')
    ax.set_xlabel('mean replication z of the collection')
axes[0].set_ylabel('density (9,999 overlap-preserving relabelings)')
axes[0].legend(fontsize=6, frameon=False, loc='upper left')
fig.tight_layout()
fig.savefig(figures / 'replication_overlap_null.png', dpi=300, bbox_inches='tight')
fig.savefig(figures / 'replication_overlap_null.pdf', bbox_inches='tight')
plt.show()

## 10 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'n_perm': N_PERM, 'seed': SEED,
          'robust': len(robust_ids), 'core': len(core_ids), 'relabel_called': len(relabel_all),
          'relabel_called_not_robust': len(relabel_only), 'relabel_programs': int(relabel_programs.nunique()),
          'replication_decision': REPLICATION_DECISION, 'lake_decision': lake_decision,
          'naming_decision_probe_balanced': NAMING_DECISION,
          'guard_median_abs_dz': float(guard_diff.median()),
          'class_claims': replacement[['claim', 'decision']].to_dict(orient='records')}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(json.dumps({k: v for k, v in record.items() if k != 'class_claims'}, indent=2))